In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[0]
sys.path.append(str(PROJECT_ROOT))

In [2]:
import pandas as pd
df = pd.read_csv("vanilla_convertibles_data_enhanced.csv")

In [3]:
from sklearn.model_selection import train_test_split


X = df.drop(columns=["price_convertible", "price_normalized"])
y = df[["price_normalized"]]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.20, random_state=42)

In [ ]:
from sklearn.preprocessing import QuantileTransformer

scaler_X = QuantileTransformer(output_distribution="normal", random_state=42)
scaler_y = QuantileTransformer(output_distribution="normal", random_state=42)

X_train = scaler_X.fit_transform(X_train)
X_val = scaler_X.transform(X_val)
X_test = scaler_X.transform(X_test)

y_train = scaler_y.fit_transform(y_train)
y_val = scaler_y.transform(y_val)

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
INPUT_DIM = X_train.shape[1]
OUTPUT_DIM = y_train.shape[1]

WIDTH = 528
DEPTHS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

BATCH_SIZE = 1024
LR = 1e-3
EPOCHS = 200
PATIENCE = 10

SEED = 42

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device} | Input: {INPUT_DIM} | Output: {OUTPUT_DIM}")

In [7]:
from torch.utils.data import TensorDataset, DataLoader

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=BATCH_SIZE)

In [ ]:
from nn_build import build_mlp

results = []

for depth in DEPTHS:
    print(f"\n{'='*40}")
    print(f"Depth={depth}, Width={WIDTH}")
    print(f"{'='*40}")

    torch.manual_seed(SEED)
    model = build_mlp(INPUT_DIM, OUTPUT_DIM, depth, WIDTH, batch_norm=True).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
    criterion = nn.MSELoss()

    best_val = float("inf")
    best_state = None
    wait = 0

    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            loss = criterion(pred, yb)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item() * len(xb)
        train_loss /= len(train_loader.dataset)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                val_loss += criterion(model(xb), yb).item() * len(xb)
        val_loss /= len(val_loader.dataset)

        if val_loss < best_val:
            best_val = val_loss
            best_state = model.state_dict().copy()
            wait = 0
        else:
            wait += 1

        if (epoch + 1) % 10 == 0 or wait == 0:
            print(f"  Epoch {epoch+1:3d} | Train: {train_loss:.6f} | Val: {val_loss:.6f} | Wait: {wait}")

        if wait >= PATIENCE:
            print(f"  Early stop at epoch {epoch+1}")
            break

    results.append({"depth": depth, "best_val_loss": best_val, "best_state": best_state})
    print(f"  Best Val Loss: {best_val:.6f}")


Depth=1, Width=528
  Epoch  10 | Train: nan | Val: nan | Wait: 10
  Early stop at epoch 10
  Best Val Loss: inf

Depth=2, Width=528
  Epoch  10 | Train: nan | Val: nan | Wait: 10
  Early stop at epoch 10
  Best Val Loss: inf

Depth=3, Width=528
